# Variable-mean noise: LN models across a luminance step

**The experiment.** `VariableMeanNoise` delivers Gaussian noise of constant contrast **through
an LED** while the mean light level steps periodically. One epoch therefore contains steps in
both directions, and the question is how the cell's linear-nonlinear model changes as it adapts
to each new mean.

The protocol lives in two packages —
`edu.washington.riekelab.rieke.protocols.VariableMeanNoise` and
`edu.washington.riekelab.turner.protocols.VariableMeanNoise` — and they are the same protocol:
the recorded epoch parameters are identical, verified against the stored epochs rather than
assumed. Both are searched together and `protocol_name` says which one a block came from.

**Only long epochs are analyzed.** The mean steps part-way through, so an epoch has to be long
enough to hold a post-step stretch worth fitting. `MIN_STIM_TIME_MS` is 30 s; the protocol's own
default is 600 ms and most recorded blocks are short runs, which §1 drops and reports.

**No filter wheel.** The LED does not sit behind the wheel, so a `FilterWheel` NDF recorded
alongside the LED's own filters does not attenuate this stimulus. §3 uses the LED's `ndfs` only
and reports the wheel separately.

Model fitting is **cascadegraph**, vendored at `retinanalysis.utils.cascadegraph` — the Python
port of the library the MATLAB used. Nothing here reimplements a filter or a sigmoid.

In [ ]:
import contextlib
import io
import json
import sys
import time
from pathlib import Path

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This notebook requires the retinanalysis Python 3.11 kernel; '
        f'got Python {sys.version.split()[0]} at {sys.executable}')
import_started = time.perf_counter()

import numpy as np
import pandas as pd
from IPython.display import display

import retinanalysis as ra
from retinanalysis.SCutils import explore as sc

# The analysis module sits beside this notebook: it is specific to this project.
sys.path.insert(0, str(Path.cwd()))
import variable_mean_noise as vmn

print(f'Python {sys.version.split()[0]} | {sys.executable}')
print(f'Imports ready in {time.perf_counter() - import_started:.2f} s')

## 1. Find experiment dates and cells

Discovery over both package copies of the protocol, keeping only blocks whose noise ran at
least `MIN_STIM_TIME_MS` (30 s). Each row is one available cell, with the protocol settings read
off the block's own first epoch rather than taken from the protocol defaults:

| column | what it is |
|---|---|
| `stim_seconds` | noise duration; several values means the cell ran more than one |
| `led` | which LED delivered the stimulus (UV, Blue, Green, Red) |
| `ndfs` | that LED's own filters — `FW` tokens appear here but are not in its path |
| `blocks` / `epochs` / `block_ids` | what there is to analyze |

`date_index` is the handle to copy into §2.

In [ ]:
PROTOCOL_LABEL = 'variable-mean noise'

protocol_blocks = vmn.find_blocks(show=True, height=0)
protocol_cells = vmn.find_protocol_cells(protocol_blocks, show=False)

print(f'\n{PROTOCOL_LABEL}: {len(protocol_cells)} cell conditions across '
      f'{protocol_cells.exp_name.nunique()} experiments')
sc.scroll_table(
    protocol_cells[['date_index', 'exp_name', 'cell_label', 'cell_type_short',
                    'blocks', 'epochs', 'stim_seconds', 'led', 'ndfs', 'block_ids']],
    height=430, num_cols=('date_index', 'blocks', 'epochs'))

## 2. Analyze one cell condition

Enter a `date_index` and `cell_label` from §1. The section is standalone after the imports: it
does its own lookup, resolves how each block was recorded, and fits the model.

**Recording type is verified per block, and blocks are not mixed.** These blocks carry no
`onlineAnalysis`, so there is no label to overrule and the amplifier decides: series resistance
above zero means whole-cell, with polarity from the sign of the current — inward (negative mean)
is `exc`, outward is `inh` — while a reading of exactly zero is confirmed against the trace,
since it also happens when compensation was never run, and only a trace that really contains
spikes is called `extracellular`.

One cell often has blocks of **more than one type**: `2021-04-27_B cell1` was held cell-attached
for one block and whole-cell for two more. Those cannot go into one LN fit — a spike rate and a
synaptic current are different quantities — so the blocks are grouped by resolved type and
`REC_TYPE` picks the group. Fitting them together is what a majority vote would silently do, and
it turns an r² of 0.58 into −0.55.

**The stimulus has to be rebuilt.** Symphony stores the noise generator's parameters and seed,
not the waveform. `gaussian_noise_stimulus` ports `GaussianNoiseGeneratorV2` step for step, with
one unavoidable dependency: MATLAB's `RandStream('mt19937ar').randn` does not match any NumPy
generator (its `rand` does — verified — but `randn` uses a different transform), so the Gaussian
draw comes from the MATLAB engine and every later step is NumPy. **This section needs the MATLAB
engine.**

One LN model is fitted per `lightMean` — a filter fitted across two means would describe
neither. Scoring follows `LNModelWrapper.m`: a random 20% of epochs is held out on each of three
rounds and variance explained is measured on those only. `r2_train` is the in-sample value;
`nl_r2` is the sigmoid's fit to the binned nonlinearity points and is **not** model performance.

In [ ]:
DATE_INDEX = 17
CELL_LABEL = 'cell1'
REC_TYPE = None         # None picks the type with the most epochs
MAX_EPOCHS = 10         # None for every epoch; each is 30-60 s
DOWNSAMPLE = 10         # 10 kHz -> 1 kHz, by block average

selected = protocol_cells[protocol_cells.date_index.eq(DATE_INDEX)
                          & protocol_cells.cell_label.eq(CELL_LABEL)]
if selected.empty:
    raise ValueError(f'date_index {DATE_INDEX} / {CELL_LABEL} is not in Section 1')
condition = selected.iloc[0]
EXP_NAME = condition.exp_name
ALL_BLOCKS = [int(b) for b in condition.block_ids.split(', ')]

print(f'{EXP_NAME} | {condition.cell_label} ({condition.cell_type_short}) | '
      f'{condition.led} | ndfs {condition.ndfs} | {condition.stim_seconds} s')

modes = pd.DataFrame([dict(block_id=b, **vmn.resolve_block_mode(EXP_NAME, b))
                      for b in ALL_BLOCKS])
modes['n_epochs'] = [len(vmn.epoch_parameters(b)) for b in modes.block_id]
display(modes[['block_id', 'rec_type', 'series_resistance_mohm',
               'mean_current_pa', 'n_epochs', 'rec_note']])

by_type = modes.groupby('rec_type').n_epochs.sum().sort_values(ascending=False)
if len(by_type) > 1:
    print(f'this cell has blocks of {len(by_type)} recording types: '
          + ', '.join(f'{t} ({n} epochs)' for t, n in by_type.items())
          + ' -- they are fitted separately')
rec_type = REC_TYPE or by_type.index[0]
BLOCK_IDS = [int(b) for b in modes.loc[modes.rec_type.eq(rec_type), 'block_id']]
print(f'\nfitting {rec_type} blocks {BLOCK_IDS}')

analysis = vmn.analyze_condition(
    EXP_NAME, BLOCK_IDS, rec_type=rec_type,
    downsample=DOWNSAMPLE, max_epochs=MAX_EPOCHS, verbose=True)
print(f'\n{analysis}')

condition_figure = vmn.plot_condition(analysis)

### 2a. The adaptation, as two numbers

Filter time-to-peak and gain, per mean level. A cell adapted to a dimmer mean should integrate
for longer and amplify more; both should fall as the mean rises.

In [ ]:
rows = []
for mean_level in analysis.light_means:
    model = analysis.ln_model[mean_level]
    rows.append({
        'lightMean': mean_level,
        'n_epochs': analysis.n_epochs[mean_level],
        'n_train': model.n_train, 'n_test': model.n_test,
        'r2': model.r2, 'r2_train': model.r2_train, 'nl_r2': model.nl_r2,
        'time_to_peak_ms': model.time_to_peak_ms,
        'peak_gain': float(np.nanmax(np.abs(model.filter))),
        'biphasic_index': model.biphasic_index,
    })
summary = pd.DataFrame(rows)
display(summary.round(3))

if len(summary) > 1:
    dim, bright = summary.iloc[0], summary.iloc[-1]
    print(f'lightMean {dim.lightMean:g} -> {bright.lightMean:g}  '
          f'({bright.lightMean / dim.lightMean:.0f}x brighter):')
    print(f'  time-to-peak {dim.time_to_peak_ms:.0f} -> {bright.time_to_peak_ms:.0f} ms')
    print(f'  peak gain    {dim.peak_gain:.3g} -> {bright.peak_gain:.3g} '
          f'({dim.peak_gain / bright.peak_gain:.1f}x lower)')

## 3. LED light level

The LED's own neutral density filters set the light level. A `FilterWheel` NDF is real — the
wheel exists on the rig — but it is **not in the LED's path**, so it must not be added to this
stimulus's attenuation. `led_attenuation` returns it separately with `wheel_ignored` set, rather
than dropping it silently, because the same metadata is correct for a Stage protocol and wrong
here.

A filter with no entry in the rig's LED table leaves `optical_density` blank and is named in
`unknown_tokens`, so an unknown filter cannot masquerade as no attenuation.

In [ ]:
light_rows = []
for (exp_name, led), group in protocol_blocks.groupby(['exp_name', 'led'], dropna=False):
    entry = vmn.led_attenuation(group.iloc[0])
    entry['n_blocks'] = len(group)
    light_rows.append(entry)
light = pd.DataFrame(light_rows)

sc.scroll_table(
    light[['exp_name', 'rig', 'led', 'led_ndfs', 'optical_density', 'attenuation',
           'wheel_tokens_ignored', 'unknown_tokens', 'n_blocks']].round(4),
    height=340, num_cols=('optical_density', 'attenuation', 'n_blocks'))

unresolved = light[light.unknown_tokens.ne('')]
print(f'{len(light)} experiment x LED combinations | '
      f'{int(light.wheel_ignored.sum())} list an FW filter that is not in the LED path '
      f'(stripped, and named in wheel_tokens_ignored)')
if len(unresolved):
    print(f'{len(unresolved)} with filters missing from the rig LED table: '
          f'{sorted(set(unresolved.unknown_tokens))}')

## 4. The saved MATLAB summary, for comparison

`matlabSummary/rodVariableMeanNoise.mat` holds the 53 cells the MATLAB analysis was run on,
with its own fitted LN models. It is **not** an input to anything above — the analysis here runs
from the recordings. It is kept so the Python population summary can be compared against the
MATLAB's once there is one.

One thing to know when matching them up: the saved `expDate` is **two days early**, across the
board. `vmn.resolve_roster_files` applies that correction and confirms each cell by label
(case-sensitively) and type; `vmn.SAVED_DATE_OFFSET_DAYS` is the constant. That is only needed
for the comparison, so it is not run here.

In [ ]:
roster = vmn.load_summary(show=True)

sc.scroll_table(
    roster[['index', 'exp_date', 'cell_label', 'cell_type', 'rec_type',
            'epoch_len_ms', 'tau_low', 'tau_high', 'is_example']].round(2),
    height=300, num_cols=('index', 'epoch_len_ms', 'tau_low', 'tau_high'))

print(f'\nsaved dates are {vmn.SAVED_DATE_OFFSET_DAYS} days early; '
      f'vmn.resolve_roster_files() corrects and confirms them when needed.')